# Phase 1 — Example pipeline run

Run the clinical NLP pipeline on bundled examples or the competition test set.

**Prerequisites:** `uv sync` from repo root. Set the notebook kernel working directory to the repo root.

In [ ]:
import json
from pathlib import Path

from src.eval.metrics import score_sample
from src.pipeline.pipeline import build_default_pipeline, write_predictions

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = Path.cwd().parent  # when cwd is notebooks/

KB_ROOT = ROOT / "data" / "kb"
pipeline = build_default_pipeline(KB_ROOT)

## Single example

In [ ]:
sample_path = ROOT / "data" / "examples" / "sample_input.txt"
gold_path = ROOT / "data" / "examples" / "sample_output.json"

text = sample_path.read_text(encoding="utf-8")
entities = pipeline.process(text)

print(f"Extracted {len(entities)} entities\n")
for ent in entities:
    print(f"{ent['type']:12} {ent['text'][:50]!r} {ent.get('candidates', [])}")

if gold_path.is_file():
    gold = json.loads(gold_path.read_text(encoding="utf-8"))
    s = score_sample(gold, entities)
    print(
        f"\nvs gold: final={s.final:.3f} "
        f"text={s.text:.3f} assertions={s.assertions:.3f} candidates={s.candidates:.3f}"
    )

## Batch run (test set)

Download inputs first:

```bash
uv run gdown --folder "https://drive.google.com/drive/folders/1GEARAJjBU3726Et4kZnPjvKGN1O7ghO3" -O data/test --remaining-ok
```

In [ ]:
input_dir = ROOT / "data" / "test" / "input"
output_dir = ROOT / "data" / "test" / "output"

if not input_dir.is_dir():
    raise FileNotFoundError(f"Missing {input_dir} — run gdown first (see cell above)")

n = write_predictions(pipeline, input_dir, output_dir)
print(f"Wrote {n} JSON files to {output_dir}")

## Package submission ZIP

In [ ]:
import shutil
import subprocess

zip_path = ROOT / "output.zip"
staging = ROOT / ".submission_staging" / "output"
staging.parent.mkdir(parents=True, exist_ok=True)
if staging.exists():
    shutil.rmtree(staging.parent)
    staging.parent.mkdir(parents=True)
shutil.copytree(output_dir, staging)

subprocess.run(
    ["zip", "-r", str(zip_path), "output"],
    cwd=staging.parent,
    check=True,
)
shutil.rmtree(staging.parent)
print(f"Created {zip_path}")